# Random per-cell circular stimulation

Single-cell optogenetic stimulation on the Moench. For every tracked cell, `RandomStimPerCellCircle` (in `custom.py`) picks one random pixel where a 7 px disk fits entirely inside the cell, once per field of view at the first stimulation frame, and stamps the same disk on every following stimulation frame. The disk centre and radius are written onto the tracks, so `exp_data.parquet` carries `stim_center_y`, `stim_center_x` and `stim_radius` without a merge step.

The notebook follows the faro [live experiment template](https://github.com/pertzlab/faro/tree/main/templates/live_experiment). The [live experiment example](https://github.com/pertzlab/faro/blob/main/examples/live_experiment.ipynb) explains every step; the [README](https://github.com/pertzlab/faro/blob/main/README.md) is the reference.

Setup: `uv sync` in this folder, then select `.venv` as the kernel.

In [ ]:
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import faro.core.utils as utils
from faro.core.controller import Controller
from faro.core.data_structures import PowerChannel, RTMSequence, SegmentationMethod, combine
from faro.core.pipeline import ImageProcessingPipeline
from faro.core.utils import events_to_dataframe
from faro.core.writers import OmeZarrWriter

## Microscope

The experiment needs a DMD, so the Moench is the default. To dry-run the notebook without the scope, use the virtual microscope instead (it has a `CyanStim` channel and a simulated DMD).

In [ ]:
from faro.microscope.pertzlab.moench import Moench

mic = Moench(None)

# Dry run on the virtual microscope:
# from vmteach import load_microscope
# from faro.microscope.simulation import UniMMCoreSimulation
# core, sim = load_microscope("optogenetic", mode="realtime")
# mic = UniMMCoreSimulation(mmc=core)
# mic.init_scope()

## Open napari

Live view, stage control and the position list come from napari-micromanager. The status widget shows the run once the controller is bound to it below.

In [ ]:
import napari
from napari_micromanager import MainWindow
from faro.widgets import ExperimentStatusWidget

viewer = napari.Viewer()
mm_widget = MainWindow(viewer, mmcore=mic.mmc)
viewer.window.add_dock_widget(mm_widget, name="napari-micromanager")

status_widget = ExperimentStatusWidget()
viewer.window.add_dock_widget(status_widget, name="experiment status", area="right")

## Experiment settings

Where results go, how long the phases are, and which channels to use. Channel `config` names must exist in the `TTL_ERK` channel group. `SEG_CHANNEL` is the index into `IMAGING_CHANNELS` that Cellpose sees.

In [ ]:
EXPERIMENT_NAME = "random_stim_per_cell_circle"
STORAGE_ROOT = r"Z:\lhinder\data\rtm_mm_data\exp_448"   # TODO: your experiment folder

START_DELAY_H = 0.0      # hours to wait before the run starts; 0 = start immediately
INTERVAL_S = 30          # seconds between frames
N_BASELINE = 10          # frames before the stimulation block
N_STIM = 10              # consecutive stimulation frames (the disks stay fixed)
N_RECOVERY = 20          # frames after the stimulation block
N_FRAMES = N_BASELINE + N_STIM + N_RECOVERY
STIM_FRAMES = list(range(N_BASELINE, N_BASELINE + N_STIM))

CHANNEL_GROUP = "TTL_ERK"
IMAGING_CHANNELS = [
    PowerChannel(config="mScarlet3", exposure=100, group=CHANNEL_GROUP, power=5),    # ERK-KTR
    PowerChannel(config="miRFP", exposure=1000, group=CHANNEL_GROUP, power=99),      # PIP
]
SEG_CHANNEL = 0                                                                       # segment on ERK-KTR
STIM_CHANNEL = PowerChannel(config="CyanStim", exposure=250, group=CHANNEL_GROUP, power=20)
OPTOCHECK_CHANNEL = PowerChannel(config="mCitrine", exposure=300, group=CHANNEL_GROUP, power=30)

print(f"{N_FRAMES} frames at {INTERVAL_S} s = {(N_FRAMES - 1) * INTERVAL_S / 60:.1f} min")
print(f"baseline frames 0..{N_BASELINE - 1}, stim frames {STIM_FRAMES[0]}..{STIM_FRAMES[-1]}, "
      f"recovery frames {STIM_FRAMES[-1] + 1}..{N_FRAMES - 1}, optocheck on the last frame")

## Set up the pipeline

Cellpose v4 segments the ERK-KTR channel, motile tracks the cells, `FE_ErkKtr` measures the cytoplasmic to nuclear ratio, and `OptoCheckFE` measures optogenetic tool expression on the reference frame. The stimulator comes from `custom.py` next to this notebook.

In [ ]:
from custom import RandomStimPerCellCircle
from faro.feature_extraction.erk_ktr import FE_ErkKtr
from faro.feature_extraction.optocheck import OptoCheckFE
from faro.segmentation.cellpose_v4 import CellposeV4
from faro.tracking.motile_tracker import TrackerMotile

segmentators = [
    SegmentationMethod(
        name="labels",
        segmentation_class=CellposeV4(gamma=0.3),
        use_channel=SEG_CHANNEL,
        save_tracked=True,
    ),
]
feature_extractor = FE_ErkKtr("labels")
optocheck = OptoCheckFE(used_mask="labels")
stimulator = RandomStimPerCellCircle(seed=0)

# search_range is the largest centroid movement between frames, in pixels.
tracker = TrackerMotile(search_range=70)

path = os.path.join(STORAGE_ROOT, EXPERIMENT_NAME)
os.makedirs(path, exist_ok=True)
pipeline = ImageProcessingPipeline(
    storage_path=path,
    segmentators=segmentators,
    feature_extractor=feature_extractor,
    feature_extractor_ref=optocheck,
    tracker=tracker,
    stimulator=stimulator,
)
ctrl = Controller(mic, pipeline, writer=OmeZarrWriter(storage_path=path))
print("results go to", path)

In [ ]:
status_widget.set_controller(ctrl)

## DMD calibration

Run this every session so the disks land on the right camera pixels. The calibration light hits the sample, so move to an empty area first. `calibrate_dmd` always runs when called; the diagnostic plots show the detected spots.

In [ ]:
mic.calibrate_dmd(STIM_CHANNEL, verbose=True, radius=4)

## Optional: check the DMD focus

Puts a checkerboard on the DMD for live view (this illuminates the sample). Press Live in napari-micromanager on the stimulation channel, refocus, then run the second cell to return the DMD to all-on.

In [ ]:
mic.dmd.checker_board(pixels=100)

In [ ]:
mic.dmd.all_on()

## Preview

Snap one frame at the current position and check the labels and the stimulation disks before committing to a long run. Without tracks the stimulator falls back to the segmentation labels, which is fine for a preview.

In [ ]:
from faro.core.pipeline import dispatch_stim_mask

channel = IMAGING_CHANNELS[SEG_CHANNEL].config
mic.mmc.setConfig(mic.resolve_group(channel), channel)
mic.mmc.snapImage()
test_img = mic.mmc.getImage()
labels_preview = segmentators[0].segmentation_class.segment(test_img)
mask_preview = dispatch_stim_mask(
    RandomStimPerCellCircle(seed=0), {"labels": labels_preview},
    {"img_shape": test_img.shape, "fov": 0, "fov_timestep": 0}, img=test_img[None],
)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(test_img, cmap="gray")
axes[0].set_title("Raw image")
axes[1].imshow(labels_preview, cmap="nipy_spectral")
axes[1].set_title(f"Labels ({labels_preview.max()} cells)")
axes[2].imshow(test_img, cmap="gray")
axes[2].imshow(np.ma.masked_where(mask_preview == 0, mask_preview), cmap="autumn", alpha=0.8)
axes[2].set_title("Stimulation disks")
for ax in axes:
    ax.axis("off")
plt.tight_layout()

## Positions

Pick the fields of view in the napari-micromanager MDA widget, then read them here.

In [ ]:
fov_positions = utils.generate_fov_positions(mic, viewer=viewer)
print(len(fov_positions), "fields of view")

## Build the event list

One phase: baseline, a block of stimulation frames, recovery, and the optocheck channel on the last frame. `apply_fov_batching` splits the fields of view into sequential batches when they do not all fit into one interval.

In [ ]:
phase = RTMSequence(
    time_plan={"interval": INTERVAL_S, "loops": N_FRAMES},
    stage_positions=fov_positions,
    channels=IMAGING_CHANNELS,
    stim_channels=[STIM_CHANNEL],
    stim_frames=STIM_FRAMES,
    ref_channels=[OPTOCHECK_CHANNEL],
    ref_frames=[-1],
    rtm_metadata={
        "phase_name": "RandomCircleStim",
        "phase_id": 0,
        "treatment_name": "Random per-cell 7px-circle stimulation",
    },
)
events = combine(phase, axis="t")
events = utils.apply_fov_batching(events, time_per_fov=4.0)
df_events = events_to_dataframe(events)
print(f"{len(events)} events over {df_events['time'].max() / 60:.1f} min")
df_events.sort_values("timestep").head()

## Validate and load

Fix every warning before you start. Validation checks the pipeline components, the event metadata, channel names, exposure limits and the DMD calibration. `load_experiment` then shows the plan in the status widget.

In [ ]:
assert ctrl.validate_events(events), "fix the warnings above before running"
ctrl.load_experiment(events, stim_mode="current")

## Run

Start from the widget or with the next cell. The kernel stays free during the run; the widget's Stop button or `handle.cancel()` aborts it.

In [ ]:
if START_DELAY_H > 0:
    print(f"waiting {START_DELAY_H} h before starting")
    time.sleep(START_DELAY_H * 3600)
handle = ctrl.start_experiment()

In [ ]:
handle.status()

## Finish

Blocks until the last frame is processed, then writes `exp_data.parquet` with all fields of view combined. The stimulation columns are already on the stimulated rows.

In [ ]:
final = handle.wait()
ctrl.finish_experiment()
mic.post_experiment()
utils.generate_exp_data_from_tracks(path)
print(f"state {final.state!r}, {final.n_frames_received} frames, "
      f"{len(final.background_errors)} background errors")

df_exp = pd.read_parquet(os.path.join(path, "exp_data.parquet"))
stim_cols = [c for c in ("stim_center_y", "stim_center_x", "stim_radius") if c in df_exp.columns]
n_stim = int(df_exp[stim_cols[0]].notna().sum()) if stim_cols else 0
print(f"{n_stim} rows carry stimulation info ({stim_cols})")
df_exp.head()